<a href="https://colab.research.google.com/github/shown5/Hands-on-Generative-AI/blob/main/chap6_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

６章言語モデルのファインチューニング

- ファインチューニング済みエンコーダーモデルを用いたテキストのトピック分類
- 現代の LLM 時代におけるエンコーダーベースモデルの役割の理解
- デコーダーモデルを使った特定のスタイルのテキスト生成
- 命令型ファインチューニングによる単一モデルでの複数タスクの解決
- 小さいGPUでもモデルを訓練できるパラメーター効率の高いファインチューニング手法
- よりすくに計算資源でモデルの推論を実行できる手法

6.1.1 データセットの特定

In [2]:
%pip install genaibook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
  Attempting uninstall: huggingface_hub
    Found exis

In [3]:
from datasets import load_dataset

#az news データセットはテキスト分類モデルのベンチマークやデータマイニング、情報検索、データストリーミングなどの研究で広く用いられている。
# 訓練用のサンプルは１２万件あり、ファインにチューニングには十分
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [4]:
# データの具体的な例を見ていこう

raw_train_datasets = raw_datasets["train"]
raw_train_datasets[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

//{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

サンプルにテキストとラベルが含まれているが、２はどのクラスを指しているのか？
これを知るにはデータセットの features とその label フィールドを見ればいい。

In [5]:
print(raw_train_datasets.features)

# results
# {'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}
# 0なら世界のニュース、1ならスポーツ、2ならビジネス、３なら科学技術のニュース

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


# 6.1.2 使用するモデルタイプの定義

## Transformer おさらい

- エンコーダーモデル：入力の意味表現を捉える
- デコーダーモデル：文章などの新しいシーケンスを出力することを目的にしたモデル。テキスト生成に最適。
- エンコーダーデコーダー型モデル：入力シーケンスを異なる出力シーケンスに変換するタスクに適している

今回は、**分類ヘッド付きエンコーダーモデル**をアプローチとして採用する。
エンコーダーモデルにシンプルな分類ネットワーク（ヘッド）を埋め込みに追加してファインチューニングする方法。

ベースモデルの要件は以下の４つ

- エンコーダベースであること
- GPU を使えば数分くらいでファインチューニングできるモデル
- 事前訓練で確かな成果を残しているもの
- 短いテキストシーケンスを処理できること

DistilBERT が良さげらしい。

6.1.4 データセットの前処理

トークナイザーは AutoTokenizer を使おう。
transformers ライブラリは入力の長さがすべて同じ出なければならないので、padding=true で使おう。


In [6]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(batch):
  return tokenizer(
      batch["text"], truncation=True, padding=True, return_tensors="pt"
  )

tokenize_function(raw_train_datasets[:2])

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101, 18431,  2571,  3504,  2646,  3293, 13395,  1006, 26665,  1007,
         26665,  1011,  2797,  5211,  3813, 18431,  2571,  2177,  1010,  1032,
          2029,  2038,  1037,  5891,  2005,  2437,  2092,  1011, 22313,  1998,
          5681,  1032,  6801,  3248,  1999,  1996,  3639,  3068,  1010,  2038,
          5168,  2872,  1032,  2049, 29475,  2006,  2178,  2112,  1997,  1996,
          3006,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 

In [7]:
# データセット内の各要素に対して関数を並列で適用するもの

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})

6.1.5 評価指標の定義

評価指標には evaluate というライブラリが使える。
文章分類の場合は以下の指標が有力な候補となる。

- 正解率
- 適合率
- 再現率
- F1 スコア

evaluate が提供する指標には compute() メソッドがある。

In [8]:
import evaluate

accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.compute(references=[0, 1, 0, 1], predictions=[1, 0, 0, 1]))


Accuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:
Accuracy = (TP + TN) / (TP + TN + FP + FN)
 Where:
TP: True positive
TN: True negative
FP: False positive
FN: False negative

{'accuracy': 0.5}


In [9]:
f1_score = evaluate.load("f1")

def conpute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)

  # 精度と F1 スコアを求める
  acc_result = accuracy.compute(references=labels, predictions=preds)
  acc = acc_result["accuracy"]

  f1_result = f1_score.compute(
      references=labels, predictions=preds, average="weighted"
  )
  f1 = f1_result["f1"]

  return {"accuracy": acc, "f1": f1}

6.1.6 モデルの訓練

DistilBERT はエンコーダーモデルなので、そのまま使うと埋め込みが得られるだけなので、分類タスクには使えない。
この埋め込みを分類ヘッドに渡す必要がある。

AutoModelForSequenceClassification でモデルを読み込み、分類ヘッドでモデルを訓練する。以下の２つの処理が行われる。
- 言語モデルのヘッドを取り外して読み込む。ここはモデルのエンコーダー部分であり、各トークンに対して埋め込みを出力する
- モデルの上にランダムに初期化された分類ヘッドを追加する。このヘッドは単なる線形層であり、プーリング埋め込みを受け取り、クラスごとの確率を出力する。

In [10]:
import torch
from transformers import AutoModelForSequenceClassification

from genaibook.core import get_device

device = get_device()
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels
).to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


モデルの初期化ができたのでいよいよ訓練開始。

In [13]:
from transformers import TrainingArguments

batch_size = 32
training_args = TrainingArguments(
    "classifier-chapter4",
    push_to_hub=True, #モデルが保存されるたびに HuggingFace にプッシュするかどうか
    num_train_epochs=2, #何回転させるか
    eval_strategy="epoch", # 評価するタイミングの指定（epoch 終了時を指定）
    per_device_train_batch_size=batch_size, # 訓練時のコアあたりのバッチサイズ
    per_device_eval_batch_size=batch_size,
)

In [ ]:
#AG News データセットからデータを取得して訓練開始。